Name: Yi-Fan Tsai\
USC ID: 4173878189

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as fc

spark = SparkSession.builder.getOrCreate()
spark

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://cli.github.com/packages stable/main amd64 Packages [343 B]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,138 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy

In [2]:
country = spark.read.json('country.json')
city = spark.read.json('city.json')
cl = spark.read.json('countrylanguage.json')

(1)select Name, GNP\
&emsp;from country\
&emsp;where Continent like '%America%' and GNP > 100000;


In [13]:
country.filter((country.Continent.like('%America%')) & (country.GNP > 100000))\
    .select('Name', 'GNP')\
    .show()

+-------------+---------+
|         Name|      GNP|
+-------------+---------+
|    Argentina| 340238.0|
|       Brazil| 776739.0|
|       Canada| 598862.0|
|     Colombia| 102896.0|
|       Mexico| 414972.0|
|United States|8510700.0|
+-------------+---------+



(2)select avg(GNP)\
&emsp;from country\
&emsp;where Continent = 'North America' and Population > 100000;

In [14]:
country.filter((country.Continent == 'North America') & (country.Population > 100000))\
    .agg(fc.avg('GNP').alias('avg_GNP'))\
    .show()

+----------+
|   avg_GNP|
+----------+
|403444.625|
+----------+



(3)select country.Name as Country, city.Name as Capital, country.GNP\
&emsp;from country join city on country.Capital = city.ID\
&emsp;where continent = "North America" and GNP > 10000\
&emsp;order by country.GNP desc\
&emsp;limit 5;

In [15]:
country.join(city, country.Capital == city.ID)\
    .filter((country.Continent == 'North America') & (country.GNP > 10000))\
    .select(country.Name.alias('Country'), city.Name.alias('Capital'), country.GNP)\
    .orderBy(fc.desc('GNP'))\
    .limit(5)\
    .show()

+-------------+-------------------+---------+
|      Country|            Capital|      GNP|
+-------------+-------------------+---------+
|United States|         Washington|8510700.0|
|       Canada|             Ottawa| 598862.0|
|       Mexico|  Ciudad de MÃ©xico| 414972.0|
|  Puerto Rico|           San Juan|  34100.0|
|    Guatemala|Ciudad de Guatemala|  19008.0|
+-------------+-------------------+---------+



(4)select Language, count(\*)\
&emsp;from countrylanguage\
&emsp;group by Language\
&emsp;having count(*) > 20;

In [16]:
cl.groupBy('Language')\
  .agg(fc.count('*').alias('count'))\
  .filter('count > 20')\
  .show()

+--------+-----+
|Language|count|
+--------+-----+
| English|   60|
| Spanish|   28|
|  French|   25|
|  Arabic|   33|
+--------+-----+



(5)select District, avg(Population)\
&emsp;from city\
&emsp;where CountryCode = "USA"\
&emsp;group by district\
&emsp;having count(*) > 10;

In [17]:
city.filter(city.CountryCode == 'USA')\
  .groupBy('District')\
  .agg(fc.avg('Population').alias('avg_Population'), fc.count('*').alias('count'))\
  .filter('count > 10')\
  .select('District', 'avg_Population')\
  .show()

+----------+------------------+
|  District|    avg_Population|
+----------+------------------+
|     Texas| 354164.6538461539|
|   Florida|210093.86666666667|
|California|245833.91176470587|
+----------+------------------+

